# 03 Document Index Validation

Kernel: **Python (NLP)**. This notebook uses only the frozen handoff files and saves derived outputs in the Phase 2 directories.

## Scope

Reload saved vectors, mapping, FAISS index, manifest, and pinned model in a fresh kernel. Fail closed if document count, IDs, dimensions, normalization, FAISS configuration, or smoke rankings disagree.

In [1]:
from pathlib import Path
import os, json, hashlib
import numpy as np
import pandas as pd
import faiss
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer

HANDOFF_ROOT=Path.cwd().resolve()
while not (HANDOFF_ROOT/"02_FROZEN_DATA").is_dir():HANDOFF_ROOT=HANDOFF_ROOT.parent
BUILD_ROOT=HANDOFF_ROOT/"Phase2"/"DocumentBuilding"
HF_HOME=HANDOFF_ROOT/"NLP"/"hf_cache";os.environ["HF_HOME"]=str(HF_HOME)
CORPUS_PATH=HANDOFF_ROOT/"02_FROZEN_DATA"/"Corpus_4192_Gaming_Documents.csv"
DOC_MANIFEST_PATH=BUILD_ROOT/"data"/"chronoplay_document_manifest_v1.csv"
EMBEDDINGS_PATH=BUILD_ROOT/"embeddings"/"chronoplay_minilm_document_embeddings_v1.npy"
INDEX_PATH=BUILD_ROOT/"indexes"/"chronoplay_minilm_faiss_index_v1.faiss"
MAPPING_PATH=BUILD_ROOT/"manifests"/"chronoplay_document_id_mapping_v1.csv"
CONFIG_PATH=BUILD_ROOT/"manifests"/"chronoplay_minilm_document_embedding_manifest_v1.json"
REPORT_PATH=BUILD_ROOT/"manifests"/"chronoplay_document_index_validation_v1.json"
MODEL_ID="sentence-transformers/all-MiniLM-L6-v2";MODEL_REVISION="1110a243fdf4706b3f48f1d95db1a4f5529b4d41";DEVICE="cpu"
assert not REPORT_PATH.exists(), "Refusing to overwrite an earlier validation report; version the output."

/Users/vaidhav/Desktop/MyProjects/NLP/ChronoPlay_Team_Handoff_v1/NLP/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1028 15:45:42.992000 9531 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [2]:
# Independently reload every artifact and the frozen document IDs.
config=json.loads(CONFIG_PATH.read_text())
corpus=pd.read_csv(CORPUS_PATH,usecols=["id"],dtype={"id":"string"})
doc_manifest=pd.read_csv(DOC_MANIFEST_PATH,dtype={"id":"string"})
mapping=pd.read_csv(MAPPING_PATH,dtype={"document_id":"string"})
embeddings=np.load(EMBEDDINGS_PATH,mmap_mode="r")
index=faiss.read_index(str(INDEX_PATH))
checks=[]
def check(name,condition,detail):
    checks.append({"check":name,"status":"PASS" if bool(condition) else "FAIL","detail":str(detail)})
check("corpus_count",len(corpus)==4192,f"{len(corpus)} corpus rows")
check("source_id_uniqueness",corpus.id.is_unique,f"{corpus.id.nunique()} unique IDs")
check("document_manifest_alignment",len(doc_manifest)==4192 and doc_manifest.id.tolist()==corpus.id.tolist(),"manifest row order and IDs match corpus")
check("mapping_completeness",len(mapping)==4192 and mapping.vector_position.tolist()==list(range(4192)),f"{len(mapping)} positions")
check("mapping_id_alignment",mapping.document_id.tolist()==corpus.id.astype(str).tolist(),"every vector position maps to its source row ID")
check("embedding_shape",embeddings.shape==(4192,384),str(embeddings.shape))
check("embedding_finite",bool(np.isfinite(embeddings).all()),"no NaN/Inf values")
norms=np.linalg.norm(embeddings,axis=1)
check("embedding_unit_norms",bool(np.allclose(norms,1.0,atol=1e-5)),f"norm range [{norms.min():.7f}, {norms.max():.7f}]")
check("faiss_count_dimension",index.ntotal==4192 and index.d==384,f"ntotal={index.ntotal}; d={index.d}")
check("faiss_exact_inner_product",type(index).__name__=="IndexFlatIP" and index.metric_type==faiss.METRIC_INNER_PRODUCT,f"type={type(index).__name__}; metric={index.metric_type}")
check("manifest_model_revision",config["model_revision"]==MODEL_REVISION and config["loaded_snapshot_commit"]==MODEL_REVISION,config["model_revision"])
check("manifest_index_alignment",config["index_count"]==index.ntotal and config["embedding_dimension"]==index.d,"manifest and loaded index agree")
print(pd.DataFrame(checks).to_string(index=False))

                      check status                                          detail
               corpus_count   PASS                                4192 corpus rows
       source_id_uniqueness   PASS                                 4192 unique IDs
document_manifest_alignment   PASS         manifest row order and IDs match corpus
       mapping_completeness   PASS                                  4192 positions
       mapping_id_alignment   PASS every vector position maps to its source row ID
            embedding_shape   PASS                                     (4192, 384)
           embedding_finite   PASS                               no NaN/Inf values
       embedding_unit_norms   PASS               norm range [0.9999999, 1.0000001]
      faiss_count_dimension   PASS                              ntotal=4192; d=384
  faiss_exact_inner_product   PASS                      type=IndexFlatIP; metric=0
    manifest_model_revision   PASS        1110a243fdf4706b3f48f1d95db1a4f5529b4d41
   m

In [3]:
# Reload the same pinned model and repeat fixed non-dataset smoke queries against the persisted index.
snapshot_dir=Path(snapshot_download(repo_id=MODEL_ID,revision=MODEL_REVISION,cache_dir=str(HF_HOME),allow_patterns=["*.json","*.txt","*.safetensors"],local_files_only=True))
import torch
torch.set_num_threads(1)
faiss.omp_set_num_threads(1)
model=SentenceTransformer(str(snapshot_dir),device=DEVICE)
smoke=[item["query"] for item in config["smoke_tests"]]
smoke_vectors=np.asarray(model.encode(smoke,convert_to_numpy=True,normalize_embeddings=True,device=DEVICE),dtype=np.float32)
scores_a,positions_a=index.search(smoke_vectors,5)
scores_b,positions_b=index.search(smoke_vectors,5)
check("reload_search_reproducible",np.array_equal(positions_a,positions_b) and np.allclose(scores_a,scores_b,atol=1e-7),"repeated top-5 IDs/scores match")
expected=[[int(x["position"]) for x in item["top5"]] for item in config["smoke_tests"]]
check("saved_smoke_results_reproduced",positions_a.tolist()==expected,"top-5 positions match embedding notebook manifest")
check("model_revision_reloaded",snapshot_dir.name==MODEL_REVISION,f"snapshot={snapshot_dir.name}")
# Record sizes and SHA256s to make copying or storage problems visible.
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):h.update(block)
    return h.hexdigest()
artifacts={}
for name,path in {"embeddings":EMBEDDINGS_PATH,"index":INDEX_PATH,"mapping":MAPPING_PATH,"config":CONFIG_PATH}.items():
    artifacts[name]={"bytes":path.stat().st_size,"sha256":sha256_file(path)}
check("saved_artifacts_exist",all(Path(p).is_file() for p in (EMBEDDINGS_PATH,INDEX_PATH,MAPPING_PATH,CONFIG_PATH)),str(artifacts))
report={"status":"PASS" if all(x["status"]=="PASS" for x in checks) else "FAIL","checks":checks,"artifacts":artifacts,"smoke_queries":smoke}
REPORT_PATH.write_text(json.dumps(report,indent=2)+"\n",encoding="utf-8")
print("\nOverall:",report["status"],"validation report:",REPORT_PATH)
assert report["status"]=="PASS","Critical index validation failed; stop before query-cleaning experiments."

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11366.08it/s]


Overall: PASS validation report: /Users/vaidhav/Desktop/MyProjects/NLP/ChronoPlay_Team_Handoff_v1/Phase2/DocumentBuilding/manifests/chronoplay_document_index_validation_v1.json
